# Unsupervised Generative AI for Brain Tumor Synthesis in Healthy Mouse Brain MRIs

## 1. Overview and Theoretical Formulation

### The Problem
Training robust deep learning models for brain tumor segmentation requires large datasets of annotated magnetic resonance images (MRIs). However, obtaining human-annotated tumor masks is expensive, extremely time-consuming, and prone to high inter-reader variability. This is especially challenging in preclinical research (e.g., mouse models) where datasets are small and expert resources are scarce.

### Proposed Solution
Rather than relying on human annotations, this notebook implements an **unsupervised generative AI pipeline** that learns the distribution of healthy mouse brain MRIs and inserts **biologically plausible synthetic tumors** with **procedurally generated ground-truth masks** into the healthy volumes. This creates pair-matched training datasets of abnormal MRI scans and matching segmentation masks, drastically reducing dependency on manually drawn segmentations.

### Adaptation of the Paper's Principles
This pipeline adapts concepts from the paper *"Unsupervised generative AI for enhancing brain tumor segmentation in multi-center, incomplete real-world data scenarios"* (inspired by architectures like UMMGAT - Unpaired Multi-Modal Group-Attention Translation):
1. **Unsupervised / Unpaired Learning:** Instead of paired image-to-image translation, the system utilizes unpaired training to model healthy-to-diseased mappings.
2. **Style and Content Separation:**
   * A **Content Encoder** encodes the background anatomy of the healthy mouse brain.
   * A **Style Encoder** captures the texture, intensity contrast, and boundary characteristics of brain lesions from a target crop.
3. **Tumor Mapper:** Maps latent noise vectors combined with target condition priors (such as necrosis, enhancement, and edema dimensions) into style embeddings.
4. **Style-Conditioned Generator:** A 3D architecture that blends healthy anatomy with the style code using **Adaptive Instance Normalization (AdaIN)**, inserting the anomaly in locations specified by a procedural tumor mask.
5. **Dual Discriminators:** A global patch-based 3D discriminator judges global slice realism, while a **Tumor-ROI crop discriminator** ensures that synthesized tumor boundaries, necrotic centers, and edema boundaries are realistic.
6. **Cycle-Consistency and Content Preservation:** Forces the generator to preserve healthy anatomy outside the anomaly region (using a background L1 mask constraint) and recover the input when no tumor is requested.

---  
**Disclaimer:** This software is designed for research, simulation, and training pipeline development only. Synthesized tumor masks represent procedural biological priors and are not validated for clinical or veterinary diagnostic purposes.

## 2. Environment Setup

To run this notebook locally, we recommend using a dedicated Conda environment. Execute the following commands in your shell:

```bash
# Create conda environment
conda create -n brats_seg python=3.10 -y
conda activate brats_seg

# Install PyTorch, MONAI, and standard scientific packages
pip install torch monai nibabel scipy scikit-image pandas matplotlib tqdm
```

In [ ]:
import os
import json
import math
import random
import csv
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import scipy.ndimage as ndimage
import skimage.morphology as morphology
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Conditional MONAI import with fallback
try:
    import monai
    from monai.losses import DiceLoss
    from monai.networks.nets import UNet as MonaiUNet
    HAS_MONAI = True
except ImportError:
    HAS_MONAI = False

# Set device
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"MONAI Available: {HAS_MONAI}")

## 3. Configuration Cell

Here we define parameters controlling paths, image preprocessing, generator architecture, and biological prior generation.

In [ ]:
@dataclass
class PipelineConfig:
    # Data Directories (TODO: Modify these relative paths to point to your local datasets)
    healthy_mri_dir: str = "./data/healthy_mri"
    brain_mask_dir: str = "./data/brain_masks"
    output_dir: str = "./data/synthetic_output"
    checkpoint_dir: str = "./checkpoints"
    
    # Optional real tumor directories (used for supervised discriminator tuning, if available)
    real_tumor_mri_dir: str = "./data/real_tumor_mri"
    real_tumor_mask_dir: str = "./data/real_tumor_masks"
    
    # Imaging & Normalization parameters
    target_spacing: tuple = (0.1, 0.1, 0.1) # mm spacing (isotropic)
    target_shape: tuple = (64, 64, 64)      # Uniform 3D shape (Depth, Height, Width)
    intensity_normalization: str = "zscore"
    
    # Generative Network Parameters
    patch_size: tuple = (32, 32, 32)        # Size for local Tumor-Aware Discriminator
    batch_size: int = 2
    learning_rate: float = 2e-4
    epochs: int = 2                         # Small default for quick demo verification; scale to 100+ for real training
    num_synthetic_tumors_per_volume: int = 2
    
    # Procedural Biological Tumor Prior parameters
    min_radius: float = 3.0                 # Minimum radius of ellipsoid seed (in voxels)
    max_radius: float = 10.0                # Maximum radius of ellipsoid seed (in voxels)
    edema_prob: float = 0.8                 # Probability of surrounding edema shell
    necrotic_core_prob: float = 0.6         # Probability of necrotic core inside tumor
    enhancement_prob: float = 0.5           # Probability of active enhancing tumor rim
    infiltrative_boundary_blur: float = 1.2 # Sigma for Gaussian blurring of boundary infiltration
    mass_effect_strength: float = 0.25      # Deformation factor simulating mechanical tissue shift
    
    # Fast-run / Debug settings
    debug_mode: bool = True

config = PipelineConfig()

# Create workspaces directories
for folder in [config.healthy_mri_dir, config.brain_mask_dir, config.output_dir, config.checkpoint_dir]:
    os.makedirs(folder, exist_ok=True)

print("Directory structure initialized.")

## 4. Preclinical Mock Data Generator

To make this notebook completely self-contained and runnable out-of-the-box, the following cell generates 5 synthetic healthy mouse brains and saves them as NIfTI volumes. Each mock volume represents a stylized brain containing lateral ventricles, cortex, subcortex, and a surrounding skull background.

In [ ]:
def create_mock_nifti_dataset(config, num_volumes=5):
    """
    Generates mock healthy 3D mouse brains and corresponding binary brain masks.
    Saves NIfTI files (.nii.gz) into the config's input directories.
    """
    shape = config.target_shape
    cx, cy, cz = shape[0] // 2, shape[1] // 2, shape[2] // 2
    
    for i in range(num_volumes):
        mri = np.zeros(shape, dtype=np.float32)
        mask = np.zeros(shape, dtype=np.uint8)
        
        # Brain ellipsoid boundary
        rx, ry, rz = shape[0] // 2 - 3, shape[1] // 2 - 4, shape[2] // 2 - 5
        x, y, z = np.ogrid[:shape[0], :shape[1], :shape[2]]
        dist = ((x - cx)/rx)**2 + ((y - cy)/ry)**2 + ((z - cz)/rz)**2
        
        mask[dist <= 1.0] = 1
        
        # Background noise
        mri += np.random.normal(0.04, 0.005, shape)
        
        # Healthy brain tissue intensities
        parenchyma = (1.0 - dist) * 0.5 + 0.35
        mri[mask == 1] = parenchyma[mask == 1]
        
        # Ventricles (low intensity fluid chambers)
        v_rx, v_ry, v_rz = 5, 6, 2.5
        v_dist_l = ((x - cx)/v_rx)**2 + ((y - (cy - 4))/v_ry)**2 + ((z - cz)/v_rz)**2
        v_dist_r = ((x - cx)/v_rx)**2 + ((y - (cy + 4))/v_ry)**2 + ((z - cz)/v_rz)**2
        
        mri[(v_dist_l <= 1.0) & (mask == 1)] = 0.12 + np.random.normal(0, 0.005, mri[v_dist_l <= 1.0].shape)[0]
        mri[(v_dist_r <= 1.0) & (mask == 1)] = 0.12 + np.random.normal(0, 0.005, mri[v_dist_r <= 1.0].shape)[0]
        
        # High-intensity outer cortex ring
        cortex = (dist > 0.85) & (dist <= 1.0)
        mri[cortex & (mask == 1)] *= 1.25
        
        # Normalizing spatial bias field
        bias = np.ones(shape, dtype=np.float32) + 0.08 * (x / shape[0])
        mri[mask == 1] *= bias[mask == 1]
        mri = np.clip(mri, 0.0, 1.5)
        
        # Save using nibabel
        mri_img = nib.Nifti1Image(mri, np.eye(4))
        mask_img = nib.Nifti1Image(mask, np.eye(4))
        
        nib.save(mri_img, os.path.join(config.healthy_mri_dir, f"mouse_{i:03d}_mri.nii.gz"))
        nib.save(mask_img, os.path.join(config.brain_mask_dir, f"mouse_{i:03d}_mask.nii.gz"))
        
    print(f"Mock dataset containing {num_volumes} healthy volumes successfully created.")

create_mock_nifti_dataset(config)

## 5. Data Loading and Preprocessing

This section implements robust intensity normalization using percentiles inside the brain mask, pad-cropping to target shape, a slice viewer, and a PyTorch `Dataset`.

In [ ]:
def robust_normalize_brain(volume, mask, lower_pct=1.0, upper_pct=99.0):
    """
    Normalize MRI intensities robustly using only voxels inside the brain mask.
    Applies percentile clipping and robust scaling (Z-score via median and IQR).
    """
    brain_voxels = volume[mask > 0]
    if len(brain_voxels) == 0:
        return (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)
        
    # Percentile clipping
    low = np.percentile(brain_voxels, lower_pct)
    high = np.percentile(brain_voxels, upper_pct)
    clipped = np.clip(volume, low, high)
    
    # Robust statistics
    median = np.median(brain_voxels)
    q25 = np.percentile(brain_voxels, 25)
    q75 = np.percentile(brain_voxels, 75)
    iqr = max(q75 - q25, 1e-6)
    
    # Scale z-score to range [-3.0, 3.0] and map to [0.0, 1.0]
    norm = (clipped - median) / iqr
    norm = np.clip(norm, -3.0, 3.0)
    norm = (norm + 3.0) / 6.0
    
    # Zero background
    norm[mask == 0] = 0.0
    return norm.astype(np.float32)

class HealthyMRIDataset(Dataset):
    def __init__(self, mri_dir, mask_dir, target_shape=(64, 64, 64)):
        self.mri_dir = Path(mri_dir)
        self.mask_dir = Path(mask_dir)
        self.target_shape = target_shape
        
        self.mri_paths = sorted(list(self.mri_dir.glob("*.nii.gz")))
        self.mask_paths = sorted(list(self.mask_dir.glob("*.nii.gz")))
        
        assert len(self.mri_paths) == len(self.mask_paths), f"Count mismatch: {len(self.mri_paths)} MRIs, {len(self.mask_paths)} masks."
        
    def __len__(self):
        return len(self.mri_paths)
        
    def __getitem__(self, idx):
        mri_path = self.mri_paths[idx]
        mask_path = self.mask_paths[idx]
        
        mri_vol = nib.load(mri_path).get_fdata().astype(np.float32)
        mask_vol = nib.load(mask_path).get_fdata().astype(np.uint8)
        
        # Preprocess and normalize
        norm_mri = robust_normalize_brain(mri_vol, mask_vol)
        
        # Transform to PyTorch tensors [Channels, D, H, W]
        mri_tensor = torch.from_numpy(norm_mri).unsqueeze(0)
        mask_tensor = torch.from_numpy(mask_vol).unsqueeze(0)
        
        # Pad/Crop to target shape if spacing or grid varies
        mri_tensor = self._match_grid(mri_tensor, self.target_shape)
        mask_tensor = self._match_grid(mask_tensor, self.target_shape, mode="nearest")
        
        return mri_tensor, mask_tensor, mri_path.name
        
    def _match_grid(self, tensor, target_shape, mode="bilinear"):
        curr = tensor.shape[1:]
        if curr == target_shape:
            return tensor
        temp = tensor.unsqueeze(0)
        if mode == "nearest":
            temp = F.interpolate(temp.float(), size=target_shape, mode="nearest")
        else:
            temp = F.interpolate(temp, size=target_shape, mode="trilinear", align_corners=False)
        return temp.squeeze(0)

def view_slices_3d(volume, label_map=None, title="", slice_indices=None):
    """
    Plots axial, coronal, and sagittal planes.
    """
    if isinstance(volume, torch.Tensor):
        volume = volume.detach().cpu().numpy().squeeze()
    if isinstance(label_map, torch.Tensor):
        label_map = label_map.detach().cpu().numpy().squeeze()
        
    d, h, w = volume.shape
    if slice_indices is None:
        slice_indices = (d // 2, h // 2, w // 2)
        
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    axes[0].imshow(volume[slice_indices[0], :, :], cmap="gray")
    if label_map is not None:
        axes[0].imshow(label_map[slice_indices[0], :, :], cmap="jet", alpha=0.3)
    axes[0].set_title(f"Axial (slice {slice_indices[0]})")
    axes[0].axis("off")
    
    axes[1].imshow(volume[:, slice_indices[1], :], cmap="gray")
    if label_map is not None:
        axes[1].imshow(label_map[:, slice_indices[1], :], cmap="jet", alpha=0.3)
    axes[1].set_title(f"Coronal (slice {slice_indices[1]})")
    axes[1].axis("off")
    
    axes[2].imshow(volume[:, :, slice_indices[2]], cmap="gray")
    if label_map is not None:
        axes[2].imshow(label_map[:, :, slice_indices[2]], cmap="jet", alpha=0.3)
    axes[2].set_title(f"Sagittal (slice {slice_indices[2]})")
    axes[2].axis("off")
    
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# Verification
ds = HealthyMRIDataset(config.healthy_mri_dir, config.brain_mask_dir, config.target_shape)
sample_mri, sample_mask, sample_name = ds[0]
print(f"Loaded: {sample_name}, shape: {sample_mri.shape}, min: {sample_mri.min():.3f}, max: {sample_mri.max():.3f}")
view_slices_3d(sample_mri, sample_mask, "Mock volume & Brain Mask")

## 6. Procedural Biological Tumor-Prior Generator

To simulate a biologically plausible tumor, we generate multiple sub-compartments: necrotic core, active enhancing rim, non-enhancing core, and infiltrative edema. The code uses random ellipsoid seeding, boundary distortion using Perlin/Gaussian noise grid deformation, and morphological operations.

In [ ]:
def generate_procedural_tumor_prior(brain_mask, config):
    """
    Creates a synthetic 3D tumor prior inside the brain mask.
    Returns:
      - mask_channels: [5, D, H, W] tensor (WT, TC, ED, NC, ER)
      - label_map: [D, H, W] integer label map:
          0: Background, 1: Necrosis, 2: Enhancing Rim, 3: Non-enhancing Core, 4: Edema
    """
    if isinstance(brain_mask, torch.Tensor):
        mask_np = brain_mask.detach().cpu().numpy().squeeze().astype(np.uint8)
    else:
        mask_np = brain_mask.astype(np.uint8)
        
    shape = mask_np.shape
    brain_indices = np.argwhere(mask_np > 0)
    
    if len(brain_indices) == 0:
        cx, cy, cz = shape[0]//2, shape[1]//2, shape[2]//2
    else:
        # Choose tumor center (restrict to brain interior to prevent clipping)
        center_idx = random.choice(brain_indices)
        cx, cy, cz = int(center_idx[0]), int(center_idx[1]), int(center_idx[2])
        
    # Ellipsoid axes radii
    rx = random.uniform(config.min_radius, config.max_radius)
    ry = random.uniform(config.min_radius, config.max_radius)
    rz = random.uniform(config.min_radius, config.max_radius)
    
    # Ellipsoid distance grid
    x, y, z = np.ogrid[:shape[0], :shape[1], :shape[2]]
    dist_grid = ((x - cx)/rx)**2 + ((y - cy)/ry)**2 + ((z - cz)/rz)**2
    
    # Irregular boundary distortion via smooth noise field
    low_res_noise = np.random.normal(0, 0.14, (8, 8, 8))
    noise_field = ndimage.zoom(low_res_noise, zoom=np.array(shape)/8.0, order=1)
    dist_grid_irregular = dist_grid + noise_field
    
    tumor_core_mask = (dist_grid_irregular <= 1.0) & (mask_np > 0)
    
    # Edema shell creation
    edema_mask = np.zeros_like(tumor_core_mask)
    if random.random() < config.edema_prob and np.sum(tumor_core_mask) > 10:
        dilation_iter = random.randint(2, 5)
        dilated_tumor = ndimage.binary_dilation(tumor_core_mask, iterations=dilation_iter)
        edema_mask = (dilated_tumor & ~tumor_core_mask) & (mask_np > 0)
        
    # Sub-compartments: Necrosis & Enhancing Rim
    necrotic_mask = np.zeros_like(tumor_core_mask)
    enhancing_mask = np.zeros_like(tumor_core_mask)
    active_core_mask = tumor_core_mask.copy()
    
    if random.random() < config.necrotic_core_prob and np.sum(tumor_core_mask) > 60:
        erosion_iter = random.randint(2, 3)
        necrotic_mask = ndimage.binary_erosion(tumor_core_mask, iterations=erosion_iter)
        active_core_mask = tumor_core_mask & ~necrotic_mask
        
        if random.random() < config.enhancement_prob and np.sum(active_core_mask) > 30:
            further_eroded = ndimage.binary_erosion(tumor_core_mask, iterations=erosion_iter + 1)
            enhancing_mask = active_core_mask & ~further_eroded
            active_core_mask = active_core_mask & ~enhancing_mask
            
    # Separate multi-channel array [5, D, H, W]
    whole_tumor = tumor_core_mask | edema_mask
    mask_channels = np.stack([
        whole_tumor.astype(np.float32),          # WT
        tumor_core_mask.astype(np.float32),       # TC
        edema_mask.astype(np.float32),            # ED
        necrotic_mask.astype(np.float32),         # NC
        enhancing_mask.astype(np.float32)         # ER
    ], axis=0)
    
    # Combined integer label map
    label_map = np.zeros(shape, dtype=np.uint8)
    label_map[edema_mask] = 4
    label_map[active_core_mask] = 3
    label_map[enhancing_mask] = 2
    label_map[necrotic_mask] = 1
    
    return torch.from_numpy(mask_channels).float(), torch.from_numpy(label_map).long()

# Demo generation
prior_channels, prior_label = generate_procedural_tumor_prior(sample_mask, config)
print(f"Prior shape: {prior_channels.shape}, unique labels: {np.unique(prior_label.numpy())}")
view_slices_3d(sample_mri, prior_label, "Preclinical MRI + Procedural Tumor Prior Mask")

## 7. Unsupervised Generative Model Design

This section builds a 3D architecture inspired by UMMGAT. We define:
1. `AdaIN3D` to modulate content maps using style parameters.
2. `ContentEncoder3D` to compress healthy anatomy.
3. `TumorStyleEncoder3D` to parse style parameters from image crops.
4. `TumorMapper` mapping random noise + parameters to style parameters.
5. `Generator3D` synthesizing the volume.
6. `Discriminator3D` (Global patch-based) & `TumorAwareDiscriminator3D` (focused on tumor crops).

In [ ]:
class AdaIN3D(nn.Module):
    def __init__(self, in_features, style_dim=64):
        super().__init__()
        self.norm = nn.InstanceNorm3d(in_features, affine=False)
        self.fc = nn.Linear(style_dim, in_features * 2)
        
    def forward(self, content, style):
        # content: [B, C, D, H, W], style: [B, style_dim]
        style_params = self.fc(style).view(style.size(0), -1, 1, 1, 1)
        gamma, beta = torch.chunk(style_params, chunks=2, dim=1)
        return (1.0 + gamma) * self.norm(content) + beta

class ConvBlock3D(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv = nn.Conv3d(in_c, out_c, kernel_size=3, stride=stride, padding=1, bias=False)
        self.norm = nn.InstanceNorm3d(out_c)
        self.act = nn.LeakyReLU(0.2, inplace=True)
        
    def forward(self, x):
        return self.act(self.norm(self.conv(x)))

class ContentEncoder3D(nn.Module):
    def __init__(self, in_c=1, base_f=8):
        super().__init__()
        self.l1 = ConvBlock3D(in_c, base_f)
        self.l2 = ConvBlock3D(base_f, base_f*2, stride=2)   # 32^3
        self.l3 = ConvBlock3D(base_f*2, base_f*4, stride=2) # 16^3
        self.l4 = ConvBlock3D(base_f*4, base_f*8, stride=2) # 8^3
        
    def forward(self, x):
        f1 = self.l1(x)
        f2 = self.l2(f1)
        f3 = self.l3(f2)
        f4 = self.l4(f3)
        return [f1, f2, f3, f4]

class TumorStyleEncoder3D(nn.Module):
    def __init__(self, in_c=1, style_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_c, 8, kernel_size=4, stride=2, padding=1),  # 16^3
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(8, 16, kernel_size=4, stride=2, padding=1),   # 8^3
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(16, 32, kernel_size=4, stride=2, padding=1),  # 4^3
            nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(32, style_dim)
        )
        
    def forward(self, x):
        return self.net(x)

class TumorMapper(nn.Module):
    def __init__(self, latent_dim=32, cond_dim=5, style_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + cond_dim, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, style_dim)
        )
        
    def forward(self, z, cond):
        return self.net(torch.cat([z, cond], dim=1))

class Generator3D(nn.Module):
    def __init__(self, content_c=1, mask_c=5, base_f=8, style_dim=64):
        super().__init__()
        self.encoder = ContentEncoder3D(content_c, base_f)
        
        self.up3 = nn.ConvTranspose3d(base_f*8, base_f*4, kernel_size=2, stride=2)
        self.adain3 = AdaIN3D(base_f*4, style_dim)
        self.conv3 = ConvBlock3D(base_f*8 + mask_c, base_f*4)
        
        self.up2 = nn.ConvTranspose3d(base_f*4, base_f*2, kernel_size=2, stride=2)
        self.adain2 = AdaIN3D(base_f*2, style_dim)
        self.conv2 = ConvBlock3D(base_f*4 + mask_c, base_f*2)
        
        self.up1 = nn.ConvTranspose3d(base_f*2, base_f, kernel_size=2, stride=2)
        self.adain1 = AdaIN3D(base_f, style_dim)
        self.conv1 = ConvBlock3D(base_f*2 + mask_c, base_f)
        
        self.final_conv = nn.Conv3d(base_f, 1, kernel_size=3, padding=1)
        
    def forward(self, healthy_mri, mask_ch, style_code):
        f1, f2, f3, f4 = self.encoder(healthy_mri)
        
        u3 = self.adain3(self.up3(f4), style_code)
        mask_ch3 = F.interpolate(mask_ch, size=u3.shape[2:], mode="nearest")
        x3 = self.conv3(torch.cat([u3, f3, mask_ch3], dim=1))
        
        u2 = self.adain2(self.up2(x3), style_code)
        mask_ch2 = F.interpolate(mask_ch, size=u2.shape[2:], mode="nearest")
        x2 = self.conv2(torch.cat([u2, f2, mask_ch2], dim=1))
        
        u1 = self.adain1(self.up1(x2), style_code)
        mask_ch1 = F.interpolate(mask_ch, size=u1.shape[2:], mode="nearest")
        x1 = self.conv1(torch.cat([u1, f1, mask_ch1], dim=1))
        
        return torch.sigmoid(self.final_conv(x1))

class Discriminator3D(nn.Module):
    def __init__(self, in_c=1, base_f=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_c, base_f, kernel_size=4, stride=2, padding=1),     # 32^3
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(base_f, base_f*2, kernel_size=4, stride=2, padding=1),  # 16^3
            nn.InstanceNorm3d(base_f*2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(base_f*2, base_f*4, kernel_size=4, stride=2, padding=1),# 8^3
            nn.InstanceNorm3d(base_f*4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(base_f*4, 1, kernel_size=3, padding=1)                  # Output patch grid
        )
        
    def forward(self, x):
        return self.net(x)

class TumorAwareDiscriminator3D(nn.Module):
    def __init__(self, in_c=1, patch_size=(32, 32, 32)):
        super().__init__()
        self.patch_size = patch_size
        self.net = nn.Sequential(
            nn.Conv3d(in_c, 8, kernel_size=3, stride=2, padding=1), # 16^3
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(8, 16, kernel_size=3, stride=2, padding=1),  # 8^3
            nn.InstanceNorm3d(16),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv3d(16, 1, kernel_size=3, padding=1)
        )
        
    def get_crop(self, vol, centers):
        B, C, D, H, W = vol.shape
        crops = []
        pd, ph, pw = self.patch_size
        for b in range(B):
            cx, cy, cz = centers[b]
            zs = max(0, min(D - pd, cx - pd//2))
            ys = max(0, min(H - ph, cy - ph//2))
            xs = max(0, min(W - pw, cz - pw//2))
            crops.append(vol[b, :, zs:zs+pd, ys:ys+ph, xs:xs+pw])
        return torch.stack(crops, dim=0)
        
    def forward(self, x, centers):
        crop = self.get_crop(x, centers)
        return self.net(crop)

## 8. Loss Functions

We formulate several constraints to regulate structure correctness and target intensity order:
1. **Content Preservation Loss:** $L_1$ outside the tumor boundary $(1 - WT)$.
2. **Cycle Consistency:** Reconstruction accuracy if zero mask & zero style code are evaluated.
3. **Style Loss:** $L_2$ between mapper style target and style features from generator's tumor patch.
4. **Intensity Order prior:** Penalizes if enhancements are darker than necrotic centers, etc.
5. **Spatial Mask Consistency:** Enforces a minimum intensity deviation inside the mask.

In [ ]:
def content_preservation_loss(gen, healthy, wt_mask):
    weight = 1.0 - wt_mask
    return torch.sum(torch.abs(gen - healthy) * weight) / (torch.sum(weight) + 1e-8)

def adversarial_loss(pred, real=True):
    target = torch.ones_like(pred) if real else torch.zeros_like(pred)
    return torch.mean((pred - target) ** 2) # LSGAN

def smoothness_loss(vol):
    d = torch.abs(vol[:, :, 1:, :, :] - vol[:, :, :-1, :, :])
    h = torch.abs(vol[:, :, :, 1:, :] - vol[:, :, :, :-1, :])
    w = torch.abs(vol[:, :, :, :, 1:] - vol[:, :, :, :, :-1])
    return torch.mean(d) + torch.mean(h) + torch.mean(w)

def intensity_order_prior(gen, label_map):
    B = gen.size(0)
    loss = 0.0
    for b in range(B):
        img = gen[b, 0]
        lbl = label_map[b]
        
        nc_mask = (lbl == 1)
        er_mask = (lbl == 2)
        tc_mask = (lbl == 3)
        
        nc_mean = torch.mean(img[nc_mask]) if torch.sum(nc_mask) > 0 else torch.tensor(0.0, device=gen.device)
        er_mean = torch.mean(img[er_mask]) if torch.sum(er_mask) > 0 else torch.tensor(0.0, device=gen.device)
        tc_mean = torch.mean(img[tc_mask]) if torch.sum(tc_mask) > 0 else torch.tensor(0.0, device=gen.device)
        
        # Relate: Enhancing Rim (ER) > Active Core (TC) > Necrosis (NC)
        if torch.sum(er_mask) > 0 and torch.sum(tc_mask) > 0:
            loss += torch.clamp(tc_mean - er_mean + 0.12, min=0.0)
        if torch.sum(tc_mask) > 0 and torch.sum(nc_mask) > 0:
            loss += torch.clamp(nc_mean - tc_mean + 0.12, min=0.0)
    return loss / B

def mask_consistency_loss(gen, healthy, wt_mask):
    diff = torch.abs(gen - healthy)
    inside = torch.sum(diff * wt_mask) / (torch.sum(wt_mask) + 1e-8)
    # Penalize if generated tumor is virtually identical to healthy tissues
    return torch.clamp(0.09 - inside, min=0.0)

## 9. Training Loop

The training cell manages generator, style encoder, and discriminator updates. It logs parameters and shows intermediate results.

In [ ]:
def extract_centers(labels):
    centers = []
    for b in range(labels.size(0)):
        idx = np.argwhere(labels[b].cpu().numpy() > 0)
        if len(idx) == 0:
            centers.append((labels.shape[1]//2, labels.shape[2]//2, labels.shape[3]//2))
        else:
            mean = idx.mean(axis=0).astype(int)
            centers.append((mean[0], mean[1], mean[2]))
    return centers

def train_pipeline(G, D, local_D, style_enc, mapper, dataloader, config):
    opt_G = torch.optim.Adam(list(G.parameters()) + list(mapper.parameters()), lr=config.learning_rate, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(list(D.parameters()) + list(local_D.parameters()), lr=config.learning_rate, betas=(0.5, 0.999))
    
    scaler_G = torch.amp.GradScaler(device=DEVICE, enabled=(DEVICE == "cuda"))
    scaler_D = torch.amp.GradScaler(device=DEVICE, enabled=(DEVICE == "cuda"))
    
    epochs = config.epochs
    print(f"Starting training for {epochs} epochs on device: {DEVICE}")
    
    for ep in range(epochs):
        G.train(); D.train(); local_D.train(); mapper.train(); style_enc.train()
        g_sum, d_sum = 0.0, 0.0
        
        for batch_idx, (mris, masks, _) in enumerate(dataloader):
            B = mris.size(0)
            mris = mris.to(DEVICE)
            masks = masks.to(DEVICE)
            
            # Generate procedural labels
            m_ch_list, label_list = [], []
            for b in range(B):
                mc, lm = generate_procedural_tumor_prior(masks[b], config)
                m_ch_list.append(mc)
                label_list.append(lm)
                
            mask_channels = torch.stack(m_ch_list, dim=0).to(DEVICE)
            labels = torch.stack(label_list, dim=0).to(DEVICE)
            wt_mask = mask_channels[:, 0:1]
            centers = extract_centers(labels)
            
            z = torch.randn(B, 32, device=DEVICE)
            cond = torch.stack([
                torch.mean(mask_channels[:, 1], dim=(1,2,3)),
                torch.mean(mask_channels[:, 2], dim=(1,2,3)),
                torch.mean(mask_channels[:, 3], dim=(1,2,3)),
                torch.mean(mask_channels[:, 4], dim=(1,2,3)),
                torch.tensor([float(config.mass_effect_strength)]*B, device=DEVICE)
            ], dim=1)
            
            # ---------------------------------------
            # Update Discriminators
            # ---------------------------------------
            opt_D.zero_grad()
            with torch.amp.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
                style = mapper(z, cond)
                fakes = G(mris, mask_channels, style)
                
                pred_r_glob = D(mris)
                pred_f_glob = D(fakes.detach())
                loss_d_glob = 0.5 * (adversarial_loss(pred_r_glob, True) + adversarial_loss(pred_f_glob, False))
                
                pred_r_loc = local_D(mris, centers)
                pred_f_loc = local_D(fakes.detach(), centers)
                loss_d_loc = 0.5 * (adversarial_loss(pred_r_loc, True) + adversarial_loss(pred_f_loc, False))
                
                loss_D = loss_d_glob + loss_d_loc
                
            scaler_D.scale(loss_D).backward()
            scaler_D.step(opt_D)
            scaler_D.update()
            d_sum += loss_D.item()
            
            # ---------------------------------------
            # Update Generator & Mapper
            # ---------------------------------------
            opt_G.zero_grad()
            with torch.amp.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
                fakes = G(mris, mask_channels, style)
                
                pred_f_glob = D(fakes)
                pred_f_loc = local_D(fakes, centers)
                loss_g_adv = 0.5 * (adversarial_loss(pred_f_glob, True) + adversarial_loss(pred_f_loc, True))
                
                loss_g_content = content_preservation_loss(fakes, mris, wt_mask) * 12.0
                
                # Style reconstruction constraint
                crops = local_D.get_crop(fakes, centers)
                pred_style = style_enc(crops)
                loss_g_style = torch.mean((pred_style - style) ** 2) * 2.0
                
                # Zero-conditioning cycle reconstruction
                zero_m = torch.zeros_like(mask_channels)
                zero_s = torch.zeros_like(style)
                restored = G(mris, zero_m, zero_s)
                loss_g_cycle = torch.mean(torch.abs(restored - mris)) * 10.0
                
                loss_g_intensity = intensity_order_prior(fakes, labels) * 1.5
                loss_g_mask_cons = mask_consistency_loss(fakes, mris, wt_mask) * 5.0
                loss_g_smooth = smoothness_loss(fakes) * 0.1
                
                loss_G = loss_g_adv + loss_g_content + loss_g_style + loss_g_cycle + loss_g_intensity + loss_g_mask_cons + loss_g_smooth
                
            scaler_G.scale(loss_G).backward()
            scaler_G.step(opt_G)
            scaler_G.update()
            g_sum += loss_G.item()
            
        print(f"Epoch {ep+1}/{epochs} -> G loss: {g_sum/len(dataloader):.4f}, D loss: {d_sum/len(dataloader):.4f}")
        
        # Visualize
        view_slices_3d(mris[0], labels[0], f"Epoch {ep+1}: Prior location overlay")
        view_slices_3d(fakes[0], title=f"Epoch {ep+1}: Synthesized result")
        view_slices_3d(torch.abs(fakes[0] - mris[0]), title=f"Epoch {ep+1}: Abs Difference Map")
        
    # Save checkpoints
    torch.save(G.state_dict(), os.path.join(config.checkpoint_dir, "generator_3d.pt"))
    torch.save(mapper.state_dict(), os.path.join(config.checkpoint_dir, "mapper.pt"))
    print("Generative checkpoints saved.")

# Init models and start training
dataloader = DataLoader(ds, batch_size=config.batch_size, shuffle=True)
G = Generator3D().to(DEVICE)
D = Discriminator3D().to(DEVICE)
local_D = TumorAwareDiscriminator3D().to(DEVICE)
style_enc = TumorStyleEncoder3D().to(DEVICE)
mapper = TumorMapper().to(DEVICE)

train_pipeline(G, D, local_D, style_enc, mapper, dataloader, config)

## 10. Synthetic Dataset Generation

After training, we pass through the healthy mouse MRI dataset and generate multiple synthetically distorted cases with custom masks, saving NIfTI volumes, label files, and JSON metadata.

In [ ]:
def execute_dataset_generation(G, mapper, dataset, config, count_per_volume=2):
    G.eval()
    mapper.eval()
    
    records = []
    loader = DataLoader(dataset, batch_size=1, shuffle=False)
    
    print(f"Generating synthetic cases to {config.output_dir}...")
    
    with torch.no_grad():
        for mris, masks, names in loader:
            mris = mris.to(DEVICE)
            masks = masks.to(DEVICE)
            stem = names[0].split("_mri")[0]
            
            for s in range(count_per_volume):
                # 1. Procedural prior
                mc, lm = generate_procedural_tumor_prior(masks[0], config)
                mc_gpu = mc.unsqueeze(0).to(DEVICE)
                
                # 2. Random style parameters
                z = torch.randn(1, 32, device=DEVICE)
                cond = torch.tensor([[
                    float(torch.mean(mc[1])),
                    float(torch.mean(mc[2])),
                    float(torch.mean(mc[3])),
                    float(torch.mean(mc[4])),
                    float(config.mass_effect_strength)
                ]], device=DEVICE)
                style = mapper(z, cond)
                
                # 3. Synthesize
                fake = G(mris, mc_gpu, style).squeeze().cpu().numpy()
                lm_np = lm.cpu().numpy()
                
                # Save naming
                mri_fname = f"{stem}_synthtumor_{s:03d}.nii.gz"
                mask_fname = f"{stem}_synthtumor_{s:03d}_mask.nii.gz"
                json_fname = f"{stem}_synthtumor_{s:03d}_metadata.json"
                
                mri_path = os.path.join(config.output_dir, mri_fname)
                mask_path = os.path.join(config.output_dir, mask_fname)
                json_path = os.path.join(config.output_dir, json_fname)
                
                # Write NIfTI file
                nib.save(nib.Nifti1Image(fake, np.eye(4)), mri_path)
                nib.save(nib.Nifti1Image(lm_np.astype(np.uint8), np.eye(4)), mask_path)
                
                # Coordinates
                indices = np.argwhere(lm_np > 0)
                coords = indices.mean(axis=0).tolist() if len(indices) > 0 else [0,0,0]
                
                # Save Metadata
                metadata = {
                    "parent_healthy_volume": names[0],
                    "synth_volume": mri_fname,
                    "synth_mask": mask_fname,
                    "center_voxel": coords,
                    "mass_effect": config.mass_effect_strength,
                    "checkpoint": "generator_3d.pt"
                }
                
                with open(json_path, "w") as f:
                    json.dump(metadata, f, indent=4)
                    
                records.append({
                    "mri_path": mri_path,
                    "mask_path": mask_path,
                    "metadata_path": json_path,
                    "stem": f"{stem}_synthtumor_{s:03d}"
                })
                
    print(f"Synthesized dataset contains {len(records)} pairs written to disk.")
    return records

dataset_records = execute_dataset_generation(G, mapper, ds, config)

## 11. Quality Control (QC)

This section checks if masks are inside the brain boundary, computes connected components (rejecting anomalies split into disconnected components far apart), measures mean absolute difference outside the tumor zone, and writes a QC CSV report.

In [ ]:
def perform_automated_qc(records, healthy_dir, mask_dir, output_dir):
    qc_log = []
    
    for rec in records:
        mri = nib.load(rec["mri_path"]).get_fdata()
        mask = nib.load(rec["mask_path"]).get_fdata()
        
        with open(rec["metadata_path"], "r") as f:
            meta = json.load(f)
            
        parent_name = meta["parent_healthy_volume"]
        healthy = nib.load(os.path.join(healthy_dir, parent_name)).get_fdata()
        brain_mask = nib.load(os.path.join(mask_dir, parent_name.replace("_mri", "_mask"))).get_fdata()
        
        # 1. Brain Containment
        spill = np.sum((mask > 0) & (brain_mask == 0))
        spill_pct = (spill / np.sum(mask > 0)) * 100 if np.sum(mask > 0) > 0 else 0.0
        contained = spill_pct < 1.0
        
        # 2. Healthy anatomy MAE
        healthy_region = (mask == 0)
        mae = np.mean(np.abs(mri[healthy_region] - healthy[healthy_region]))
        
        # 3. Component topology check
        _, comp_count = ndimage.label(mask > 0)
        valid_topology = comp_count >= 1 and comp_count <= 3
        
        status = "PASS" if (contained and mae < 0.15 and valid_topology) else "FAIL"
        
        qc_log.append({
            "case_id": rec["stem"],
            "brain_spill_pct": spill_pct,
            "healthy_mae": float(mae),
            "components": comp_count,
            "status": status
        })
        
    csv_path = os.path.join(output_dir, "qc_report.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["case_id", "brain_spill_pct", "healthy_mae", "components", "status"])
        writer.writeheader()
        writer.writerows(qc_log)
        
    print(f"QC finished. CSV written to: {csv_path}")
    df = pd.DataFrame(qc_log)
    if 'display' in globals():
        display(df)
    else:
        print(df)

perform_automated_qc(dataset_records, config.healthy_mri_dir, config.brain_mask_dir, config.output_dir)

## 12. Downstream Segmentation Experiment

In this section, we train a 3D U-Net (MONAI network with fallback) on our synthesized dataset, predicting whole-tumor boundaries from the synthesized volume, and computing dice metrics.

In [ ]:
class SyntheticSegDataset(Dataset):
    def __init__(self, records):
        self.records = records
        
    def __len__(self):
        return len(self.records)
        
    def __getitem__(self, idx):
        rec = self.records[idx]
        mri = nib.load(rec["mri_path"]).get_fdata().astype(np.float32)
        mask = nib.load(rec["mask_path"]).get_fdata().astype(np.float32)
        
        wt = (mask > 0).astype(np.float32)
        return torch.from_numpy(mri).unsqueeze(0), torch.from_numpy(wt).unsqueeze(0)

class Basic3DUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.down1 = nn.Conv3d(1, 8, kernel_size=3, stride=2, padding=1)
        self.down2 = nn.Conv3d(8, 16, kernel_size=3, stride=2, padding=1)
        self.up2 = nn.ConvTranspose3d(16, 8, kernel_size=2, stride=2)
        self.up1 = nn.ConvTranspose3d(8, 1, kernel_size=2, stride=2)
        
    def forward(self, x):
        x1 = F.relu(self.down1(x))
        x2 = F.relu(self.down2(x1))
        u2 = F.relu(self.up2(x2))
        return torch.sigmoid(self.up1(u2))

def run_downstream_segmentation(records, device):
    ds_seg = SyntheticSegDataset(records)
    loader = DataLoader(ds_seg, batch_size=2, shuffle=True)
    
    if HAS_MONAI:
        model = MonaiUNet(
            spatial_dims=3,
            in_channels=1,
            out_channels=1,
            channels=(8, 16, 32),
            strides=(2, 2),
            num_res_units=1
        ).to(device)
        loss_fn = DiceLoss(sigmoid=True)
    else:
        model = Basic3DUNet().to(device)
        loss_fn = lambda pred, target: nn.BCELoss()(pred, target)
        
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    print("Training downstream segmentation model on synthesized pairs...")
    model.train()
    for epoch in range(3):
        ep_loss = 0.0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            loss.backward()
            opt.step()
            ep_loss += loss.item()
        print(f"Seg Epoch {epoch+1} Loss: {ep_loss/len(loader):.4f}")
        
    # Evaluator metrics
    model.eval()
    dices = []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = (model(x) > 0.5).float()
            intersect = torch.sum(preds * y)
            union = torch.sum(preds) + torch.sum(y)
            dice = (2.0 * intersect) / (union + 1e-8)
            dices.append(dice.item())
            
    print(f"Evaluation results: Mean Dice Score = {np.mean(dices):.4f}")
    torch.save(model.state_dict(), os.path.join(config.checkpoint_dir, "downstream_segmentation_model.pt"))

run_downstream_segmentation(dataset_records, DEVICE)

## 13. Unsupervised Anomaly Detector Section

As an alternative approach, we train a healthy-only 3D Autoencoder. Because it is trained strictly on healthy brains, it fails to reconstruct tumor lesions. The residual error map can be calibrated using synthetic masks to detect anomaly locations.

In [ ]:
class HealthyAutoencoder3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=3, stride=2, padding=1),  # 32^3
            nn.ReLU(),
            nn.Conv3d(8, 16, kernel_size=3, stride=2, padding=1), # 16^3
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(16, 8, kernel_size=2, stride=2),   # 32^3
            nn.ReLU(),
            nn.ConvTranspose3d(8, 1, kernel_size=2, stride=2),    # 64^3
            nn.Sigmoid()
        )
        
    def forward(self, x):
        return self.decoder(self.encoder(x))

def evaluate_anomaly_detector(healthy_dataset, records, device):
    ae = HealthyAutoencoder3D().to(device)
    opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
    loader = DataLoader(healthy_dataset, batch_size=2, shuffle=True)
    
    ae.train()
    for epoch in range(3):
        ep_loss = 0.0
        for x, _, _ in loader:
            x = x.to(device)
            opt.zero_grad()
            rec = ae(x)
            loss = F.mse_loss(rec, x)
            loss.backward()
            opt.step()
            ep_loss += loss.item()
        print(f"AE Epoch {epoch+1} MSE: {ep_loss/len(loader):.4f}")
        
    # Test on single synthetic volume
    ae.eval()
    test_rec = records[0]
    synth_mri = nib.load(test_rec["mri_path"]).get_fdata().astype(np.float32)
    synth_mask = nib.load(test_rec["mask_path"]).get_fdata().astype(np.float32)
    
    in_t = torch.from_numpy(synth_mri).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        rec_mri = ae(in_t).squeeze().cpu().numpy()
        
    # Calculate residual error map
    error_map = np.abs(synth_mri - rec_mri)
    
    in_tumor = error_map[synth_mask > 0]
    in_healthy = error_map[synth_mask == 0]
    
    threshold = (np.mean(in_tumor) + np.mean(in_healthy)) / 2.0
    detected = error_map > threshold
    
    tpr = np.sum(detected & (synth_mask > 0)) / (np.sum(synth_mask > 0) + 1e-8)
    fpr = np.sum(detected & (synth_mask == 0)) / (np.sum(synth_mask == 0) + 1e-8)
    
    print(f"\nAnomaly detector calibrated threshold: {threshold:.4f}")
    print(f"True Positive Rate (TPR): {tpr:.4f}")
    print(f"False Positive Rate (FPR): {fpr:.4f}")
    
    view_slices_3d(error_map, title="Residual Anomaly Map")

evaluate_anomaly_detector(ds, dataset_records, DEVICE)

## 14. Biological Plausibility and Validation Guidelines

### Essential Caveats
1. **Plausibility vs Reality:** Synthetic tumors are generated based on mathematical ellipsoids, deformed structures, and style constraints. They do NOT replace authentic tumor models unless validated against real mouse models (e.g. glioblastoma GL261 tumor lines).
2. **Procedural Bias:** Generative models are highly susceptible to learning patterns directly from the procedural inputs. Downstream classifiers must be carefully checked with real scans to ensure generalization.
3. **Expert Validation:** Synthesized images should be evaluated using radiologists or mouse MRI experts to verify if visual artifacts or borders match real scans.
4. **No Leakage:** Keep dataset partitions (train, validation, test) strictly animal-level. Do not mix synthesized tumor variations from the same subject across splits.